In [36]:
import pandas as pd
import random
import math

In [37]:
diabetes = pd.read_csv("../dataset/diabetes.csv")

diabetes.head()

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,148,72,35,0,33.6,0.627,50,1
1,1,85,66,29,0,26.6,0.351,31,0
2,8,183,64,0,0,23.3,0.672,32,1
3,1,89,66,23,94,28.1,0.167,21,0
4,0,137,40,35,168,43.1,2.288,33,1


In [38]:
print("Rows, Columns:",diabetes.shape)

Rows, Columns: (768, 9)


In [39]:
dataset = diabetes.values.tolist()

for row in dataset[:5]:
    print(row)

[6.0, 148.0, 72.0, 35.0, 0.0, 33.6, 0.627, 50.0, 1.0]
[1.0, 85.0, 66.0, 29.0, 0.0, 26.6, 0.351, 31.0, 0.0]
[8.0, 183.0, 64.0, 0.0, 0.0, 23.3, 0.672, 32.0, 1.0]
[1.0, 89.0, 66.0, 23.0, 94.0, 28.1, 0.167, 21.0, 0.0]
[0.0, 137.0, 40.0, 35.0, 168.0, 43.1, 2.288, 33.0, 1.0]


In [40]:
# Dataset Split Function

def split_dataset(dataset):

    train_set = []
    val_set = []
    test_set = []

    for sample in dataset:

        R = random.random()

        if R <= 0.70:
            train_set.append(sample)

        elif R <= 0.85:
            val_set.append(sample)

        else:
            test_set.append(sample)

    return train_set, val_set, test_set

train_set, val_set, test_set = split_dataset(dataset)

print("Training:",len(train_set))
print("Validation:",len(val_set))
print("Test:",len(test_set))

Training: 526
Validation: 110
Test: 132


In [41]:
# Euclidean Distance Function

def euclidean_distance(sample1,sample2):

    distance = 0

    # ignore output column
    for i in range(len(sample1)-1):

        distance += (
            sample1[i]-sample2[i]
        )**2

    return math.sqrt(distance)

print(euclidean_distance(train_set[0],train_set[1]))

66.90348403483932


In [42]:
# KNN Regression Function

def knn_regression(
    train_set,
    validation_sample,
    K
):

    distances = []

    for training_sample in train_set:

        distance = euclidean_distance(
            validation_sample,
            training_sample
        )

        distances.append(
            (
                distance,
                training_sample[-1]
            )
        )

    distances.sort()

    neighbors = distances[:K]

    outputs = []

    for neighbor in neighbors:

        outputs.append(
            neighbor[1]
        )

    prediction = (
        sum(outputs)/K
    )

    return prediction

In [43]:
# Test Regression Function

prediction = knn_regression(train_set,val_set[0],5)

print("Predicted:",prediction)

print("Actual:",val_set[0][-1])

Predicted: 0.8
Actual: 1.0


In [44]:
# Mean Squared Error Function

def calculate_mse(train_set,validation_set,K):

    error = 0

    for sample in validation_set:

        predicted = knn_regression(train_set,sample,K)

        actual = sample[-1]

        error += (actual-predicted)**2

    mse = (error/len(validation_set))

    return mse

In [45]:
# Validation Experiment

K_values = [1,3,5,10,15]

results = []

for K in K_values:

    mse = calculate_mse(train_set,val_set,K)

    results.append([K,mse])

    print("K =",K,"| MSE =",round(mse,4))

K = 1 | MSE = 0.2909
K = 3 | MSE = 0.2354
K = 5 | MSE = 0.1989
K = 10 | MSE = 0.1728
K = 15 | MSE = 0.1709


In [46]:
# Results Table

results_df = pd.DataFrame(results,columns=["K","Validation MSE"])

results_df

,K,Validation MSE
0,1,0.290909
1,3,0.235354
2,5,0.198909
3,10,0.172818
4,15,0.170949


In [47]:
# Find Best K

best_row = min(results,key=lambda x:x[1])

best_K = best_row[0]
best_mse = best_row[1]

print("Best K:",best_K)

print("Best Validation MSE:",round(best_mse,4))

Best K: 15
Best Validation MSE: 0.1709


In [48]:
# Final Test MSE

test_mse = calculate_mse(train_set,test_set,best_K)

print("Final Test MSE:",round(test_mse,4))

Final Test MSE: 0.1777
